In [12]:
import duckdb
import geopandas as gpd
from pathlib import Path

In [13]:
ROOT = Path.cwd().parent
RAW_DIR = ROOT / "data/raw"
PROCESSED_DIR = ROOT / "data/processed"

RIVER_PARQUET = PROCESSED_DIR / "lawa_river_water_quality.parquet"

In [14]:
con = duckdb.connect()
con.install_extension("spatial")
con.load_extension("spatial")

Check number of entries

In [15]:
sql = f""" 
    SELECT COUNT(*) AS n_rows
    FROM '{RIVER_PARQUET}'
    """
    
rel = con.sql(sql)
rel.show()

┌─────────┐
│ n_rows  │
│  int64  │
├─────────┤
│ 1842772 │
└─────────┘



Check Number of entries for Ammonium Measurements

In [16]:
sql = f"""
    SELECT COUNT(*) AS n_rows
    FROM '{RIVER_PARQUET}'
    WHERE INDICATOR = 'Ammoniacal nitrogen'
"""

n = con.sql(sql).fetchall()[0][0]

print(f'Number of ammonium measurements: {n:,}')

Number of ammonium measurements: 188,741


In [17]:
sql = f"""
    SELECT 
        LawaSiteID, 
        SiteID,
        ST_AsWKB(ANY_VALUE(geometry)) AS geometry,
        MAX(Value) AS NH4max,
        MIN(Value) AS NH4min,
        AVG(Value) AS NH4mean,
        MEDIAN(Value) AS NH4median
    FROM '{RIVER_PARQUET}'
    WHERE INDICATOR = 'Ammoniacal nitrogen'
    GROUP BY LawaSiteID, SiteID
"""

df = con.sql(sql).df()
gdf = gpd.GeoDataFrame(
    df,
    # DuckDb returns bytearray, convert to bytes first
    geometry=gpd.GeoSeries.from_wkb(df['geometry'].apply(bytes)),  
    crs='EPSG:4326'   
)

print(f'Number of LAWAsites: {len(gdf)}')
print(gdf.crs)
gdf.head()



Number of LAWAsites: 1130
EPSG:4326


,LawaSiteID,SiteID,geometry,NH4max,NH4min,NH4mean,NH4median
0,es-00011,Oreti River at Lumsden Bridge,POINT (168.42777 -45.71621),0.025,0.010,0.010161,0.0100
1,hrc-00081,Manawatu at us Fonterra Longburn,POINT (175.55681 -40.39832),0.408,0.001,0.107328,0.0925
2,hbrc-00073,1932,POINT (176.28076 -39.00064),0.049,0.005,0.007310,0.0050
3,hrc-00024,Mangatainoka at Brewery - S.H.2 Bridge,POINT (175.86191 -40.42268),0.079,0.001,0.013143,0.0100
4,hrc-00031,Ohau at Haines Property,POINT (175.19784 -40.66429),0.110,0.001,0.011351,0.0100


In [28]:
sql = f"""
    SELECT 
        LawaSiteID, 
        SiteID,
        ST_AsWKB(ANY_VALUE(geometry)) AS geometry,
        MAX(Value) AS NH4max,
        MIN(Value) AS NH4min,
        AVG(Value) AS NH4mean,
        MEDIAN(Value) AS NH4median
    FROM '{RIVER_PARQUET}'
    WHERE INDICATOR = 'Ammoniacal nitrogen'
    GROUP BY LawaSiteID, SiteID
    ORDER BY MAX(Value) DESC 
    LIMIT 5
"""

df = con.sql(sql).show()

┌─────────────┬────────────────────────────────────────┬───────────────────────────────────────────────────────────────────────┬────────┬────────┬────────────────────┬───────────┐
│ LawaSiteID  │                 SiteID                 │                               geometry                                │ NH4max │ NH4min │      NH4mean       │ NH4median │
│   varchar   │                varchar                 │                                 blob                                  │ double │ double │       double       │  double   │
├─────────────┼────────────────────────────────────────┼───────────────────────────────────────────────────────────────────────┼────────┼────────┼────────────────────┼───────────┤
│ lawa-103990 │ Moawhango at Waiouru                   │ \x01\x01\x00\x00\x00\x0DT\xC6\xBF\xCF\xF8e@\x84d\x01\x13\xB8\xB7C\xC0 │   50.6 │    0.0 │ 0.3645352112676053 │     0.006 │
│ lawa-101952 │ Piakatutu at d/s Sanson STP            │ \x01\x01\x00\x00\x00\x15m3\xBA(\xEDe@\x07\x

## Save to GeoJSON

In [18]:
gdf.to_file(PROCESSED_DIR / "ammonium.geojson", driver='GeoJSON')

## Create Map with Tippecanoe

In [19]:
!tippecanoe --version

tippecanoe v2.82.0


In [20]:
!tippecanoe -o ../ammonium.pmtiles -l ammonium -zg -r1 --force ../data/processed/ammonium.geojson

../data/processed/ammonium.geojson:235: null geometry (additional not reported): in JSON object {"type":"Feature","properties":{"LawaSiteID":"lawa-103779","SiteID":"TAWHITI at Duffy's","NH4max":0.072,"NH4min":0.005,"NH4mean":0.017222222222222223,"NH4median":0.01},"geometry":null}
1122 features, 47449 bytes of geometry and attributes, 69212 bytes of string pool, 0 bytes of vertices, 0 bytes of nodes
Choosing a maxzoom of -z5 for features typically 34883 feet (10633 meters) apart, and at least 3819 feet (1164 meters) apart
  93.7%  5/31/19  


## Setup for Browser

In [ ]:
# Python option
# %pip install rangehttpserver

In [22]:
from pmtiles.reader import Reader, MmapSource
import json

with open('../ammonium.pmtiles', 'rb') as f:
    reader = Reader(MmapSource(f))
    print(reader.header())
    print(json.dumps(reader.metadata(), indent=2))

{'version': 3, 'root_offset': 127, 'root_length': 85, 'metadata_offset': 212, 'metadata_length': 2309, 'leaf_directory_offset': 2521, 'leaf_directory_length': 0, 'tile_data_offset': 2521, 'tile_data_length': 293037, 'addressed_tiles_count': 15, 'tile_entries_count': 15, 'tile_contents_count': 15, 'clustered': True, 'internal_compression': <Compression.GZIP: 2>, 'tile_compression': <Compression.GZIP: 2>, 'tile_type': <TileType.MVT: 1>, 'min_zoom': 0, 'max_zoom': 5, 'min_lon_e7': -1767508960, 'min_lat_e7': -466297110, 'max_lon_e7': 1784275530, 'max_lat_e7': -350104560, 'center_zoom': 5, 'center_lon_e7': 1743750000, 'center_lat_e7': -364660300}
{
  "name": "../ammonium.pmtiles",
  "format": "pbf",
  "type": "overlay",
  "description": "../ammonium.pmtiles",
  "version": "2",
  "tippecanoe_decisions": {
    "basezoom": 5,
    "droprate": 1,
    "retain_points_multiplier": 1
  },
  "generator": "tippecanoe v2.82.0",
  "generator_options": "tippecanoe -o ../ammonium.pmtiles -l ammonium -zg -

For starting Python server `python -m RangeHTTPServer 8080`